# 10 - `segpt-eval`: compare finished runs on one scale

Issue #109 (map #108). Two parts: (1) the pieces on **synthetic** data, so you can see what each step does and run it anywhere; (2) the real command on finished runs.

Why a separate evaluation step? Each training run reports its own metric at its own threshold. To say "model B beats model A" you need the same IoU and object code, the same val patches, and a threshold that was not tuned on the numbers you report.

In [ ]:
import numpy as np, torch
from segmentation_tools_pt.eval import (patch_scores, aggregate, object_f1, score_dataset, split_halves, held_out,
                                        run_summary, arm_summary, compare_runs, format_table, arm_name)

## 1. One patch, any classes

`patch_scores` takes boolean `(K, H, W)` masks and the class names; nothing is chip/lifted specific.

In [ ]:
gt = np.zeros((3, 32, 32), bool); gt[0, :16] = True; gt[1, 20:24, 4:8] = True; gt[2, 28, 28] = True
pr = gt.copy(); pr[1] = False; pr[2, 2, 2] = True         # class 1 missed, class 2: one false alarm and the real one found
patch_scores(pr, gt, ('background_like', 'blob', 'dot'))

## 2. Threshold sweep with a stub model

`score_dataset` runs the model once and thresholds the probabilities at every sweep point. Here the "model" returns noisy logits of the truth, which is enough to see the F1 move with the threshold.

In [ ]:
rng = np.random.default_rng(0)
class Noisy(torch.nn.Module):
    def forward(self, x): return x
ds = []
for i in range(40):
    m = (rng.random((2, 24, 24)) > 0.97)
    logits = torch.tensor(np.where(m, 1.5, -1.5) + rng.normal(0, 1.2, m.shape), dtype=torch.float32)
    ds.append(dict(image=logits, mask=torch.tensor(m, dtype=torch.float32), sample_id=f'src{i // 4}_p{i % 4}'))
rows, _ = score_dataset(Noisy(), ds, [0.3, 0.5, 0.7, 0.9], ('a', 'b'))
for t, r in rows.items():
    a = aggregate(r.values(), ('a', 'b'))
    print(t, round(object_f1(a['b_detected'], a['b_total'], a['b_false_alarms']), 3))

## 3. Held-out threshold choice

Source images are split into two halves; the threshold is picked on one half and scored on the other, both ways.

In [ ]:
stem_source = {s: s.split('_')[0] for s in rows['0.5']}
halves = split_halves(stem_source.values())
d = held_out(rows, stem_source, halves, ('a', 'b'))
[(x['threshold'], round(x['f1_b'], 3)) for x in d]

## 4. An RF-DETR run in the same table

An RF-DETR run (`model.source: rfdetr`) is an instance model: it returns objects with scores, not a probability map. `score_rfdetr` predicts each patch **once** at the lowest swept score and cuts the detections at every threshold, so its rows look exactly like the U-Net rows and everything above (held-out choice, table) works unchanged. Here a stand-in for the `RFDETR` model so it runs without a GPU or the `rfdetr` package; with real runs `segpt-eval` does the loading (`rfdetr.load_rfdetr_predictor(run_dir)`).

In [ ]:
from types import SimpleNamespace
from segmentation_tools_pt.rfdetr import RFDETRPredictor, RFDETR_THRESHOLDS
from segmentation_tools_pt.eval import score_rfdetr

class FakeRFDETR:
    "Finds the true blob of each class with a score that grows with its size, plus one weak false alarm."
    def __init__(self, truth): self.truth = truth
    def predict(self, img, threshold=0.5):
        gt = self.truth[int(np.asarray(img)[0, 0])]            # the image encodes the patch index
        det = [(k, 0.3 + 0.1 * k, gt[k]) for k in range(2) if gt[k].any()]
        fa = np.zeros_like(gt[0]); fa[0, 0] = True
        det.append((1, 0.04, fa))
        det = [x for x in det if x[1] >= threshold]
        return SimpleNamespace(mask=np.stack([x[2] for x in det]) if det else np.zeros((0,) + gt[0].shape, bool),
                               class_id=np.array([x[0] for x in det], int), confidence=np.array([x[1] for x in det]))

truth = [np.stack([np.pad(np.ones((6, 6), bool), ((2, 24), (2, 24))), np.pad(np.ones((3, 3), bool), ((20, 9), (20, 9)))]) for _ in range(6)]
pds = [dict(image=torch.zeros(1, 32, 32), mask=torch.tensor(t, dtype=torch.float32), sample_id=f'src{i // 2}_p{i % 2}') for i, t in enumerate(truth)]
pred = RFDETRPredictor(FakeRFDETR(truth), 2)
img_for = lambda sid: np.full((32, 32), int(sid.split('_p')[0][3:]) * 2 + int(sid[-1]), np.uint8)
rf_rows, _ = score_rfdetr(pred, pds, RFDETR_THRESHOLDS, ('a', 'b'), image_for=img_for)
for t, r in rf_rows.items():
    a = aggregate(r.values(), ('a', 'b'))
    print(t, round(object_f1(a['b_detected'], a['b_total'], a['b_false_alarms']), 3))      # F1 falls as the weak false alarm gets in (low t) or the class is dropped (high t)

With real runs:

```bash
segpt-eval outputs/resnet18_e15_s42 outputs/rfdetr_small_s42 --heldout --out eval_out
```

Each run uses its own default grid (U-Net probabilities `0.05 ... 0.9999`, RF-DETR scores `0.01 ... 0.7`); `--thresholds` overrides it for all runs, and the held-out threshold is chosen per run. Run it in the environment that has `rfdetr`.

## 5. The real command

On finished runs (directories with `.hydra/config.yaml` and `checkpoints/best.pth`):

```bash
segpt-eval outputs/confirm_106/resnet18_e15_s42 outputs/confirm_106/resnet18_e15_s43 \
           outputs/confirm_106/convnext_tiny_e15_s42 outputs/confirm_106/convnext_tiny_e15_s43 \
           --heldout --out eval_out
```

It prints one table (one row per arm, mean `+-` seed spread) and writes `eval_out/<run>.json`, `<run>_per_patch.json`, `comparison.json`. If the val-id fingerprints differ it stops with exit code 1. The next cell is the same thing from Python; it needs the real cache, so it is not run here.

In [ ]:
# needs the real cache and runs; set `base` to your runs
from pathlib import Path
from segmentation_tools_pt.eval import run_eval
base = Path('outputs/confirm_106')
if base.exists():
    run_eval([base / f'{a}_e15_s{s}' for a in ('resnet18', 'convnext_tiny') for s in (42, 43)], heldout=True, out='eval_out')